# Install Libraries

In [ ]:
!pip install qdrant-client sentence-transformers google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 4.5 MB/s eta 0:00:00


# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Step 1: Configure API Keys

Before running the notebook, you need to set up your **Google Colab Secrets** so the code can access Qdrant and Gemini.

1. Click the **key icon** on the left sidebar in Colab.  
2. Add each API name and its corresponding key value:  
   - `QDRANT_URL`  
   - `QDRANT_API_KEY`  
   - `GOOGLE_API_KEY`  
3. Toggle **Notebook access** → it should turn blue with a check mark.  
4. In the notebook, import `userdata` from `google.colab`.  
5. Use `userdata.get("API_NAME")` to securely retrieve each key.  


In [ ]:
from google.colab import userdata
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Step 2: Configure Google GenAI
- Import the **Google Generative AI** library.  
- Configure it using your **Google API Key** so we can access Gemini models.  


In [ ]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

# Step 3: Configure Qdrant URL and API
- Qdrant Cloud requires the REST port `:6333` in the URL.  
- The code ensures the port is present.  
- Then we connect to Qdrant by passing the **API Key** and **URL**.  
- Finally, we do a quick test (`get_collections()`) to confirm the connection works.  


In [ ]:
# Ensure REST Port
if QDRANT_URL and QDRANT_URL.endswith(".qdrant.io"):
    if ":6333" not in QDRANT_URL:
        QDRANT_URL = QDRANT_URL.rstrip("/") + ":6333"

In [ ]:
# Connect to Qdrant
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=30,
)
try:
    _ = client.get_collections()
except Exception as e:
    raise RuntimeError(f"Qdrant connectivity failed. Check QDRANT_URL/API key/egress. Details: {e}")

# Step 4: Configure Embedding Model
- Make sure you have added your **Hugging Face Token (HF_TOKEN)** in Google Colab Secrets.  
- This token allows Colab to download the embedding model from Hugging Face.  
- The embedding model will:
  - Convert text in our knowledge base into **embedding vectors** (numbers that capture meaning).  
  - Let us check the **dimension size** of these embeddings (needed by Qdrant).  


In [ ]:
# Define Knowledge Base
knowledge_base = [
    # Institut Teknologi Sepuluh Nopember (ITS) General
    "Institut Teknologi Sepuluh Nopember (ITS) is a public university located in Surabaya, East Java, Indonesia, with campuses in Surabaya and Sidoarjo.",
    "ITS was founded on November 10, 1957, originally named Perguruan Tinggi Teknik Sepuluh Nopember Surabaya.",
    "The main campus in Sukolilo spans about 180–189 hectares, with additional campuses at Manyar and Cokroaminoto.",
    "ITS emphasizes science, engineering, technology education, and sustainable innovations in green technology and clean energy.",
    "The current rector (2024–2029) is Prof. Ir. Bambang Pramujati, S.T., M.Sc.Eng., Ph.D.",

    # ITS Informatics Department
    "The Informatics Department (Departemen Teknik Informatika) belongs to the Faculty of Intelligent Electrical and Informatics Technology (FT-EIC).",
    "Head of the Informatics Department is Prof. Ir. Ary Mazharuddin Shiddiqi, S.Kom., M.Comp.Sc., Ph.D.",
    "The Secretary of the Informatics Department is Dr.Eng. Radityo Anggoro, S.Kom., M.Sc.",
    "Study program heads: Informatics Engineering (Hadziq Fabroyir, S.Kom., Ph.D.), Software Engineering (Prof. Daniel Oranova Siahaan, S.Kom., M.Sc., PD.Eng.), Artificial Intelligence Engineering (Dr. Anny Yuniarti, S.Kom., M.Comp.Sc.), Graduate Program (Hudan Studiawan, S.Kom., M.Kom, Ph.D.).",
    "The Informatics Department offers Bachelor (S1), Master (S2), and Doctoral (S3) programs with international accreditations and strong focus on software, artificial intelligence, and computer science.",
    "The department has several research labs, including the Software Engineering Lab, led by Prof. Umi Laili Yuhana.",

    # ITS FT-EIC Faculty
    "The Faculty of Intelligent Electrical and Informatics Technology (FT-EIC) was formed by merging the former Faculty of Information and Communication Technology (FTIK) and Faculty of Electrical Technology (FTE).",
    "FT-EIC consists of 6 departments and 10 undergraduate programs with over 2,500 students across S1, S2, and S3 levels.",
    "The vision of FT-EIC is to be a leading faculty in education and research in electrical technology, electronics, information systems, biomedical, computer science, and informatics with an international reputation.",
    "The Dean of FT-EIC for 2025–2029 is Prof. Dr. Diana Purwitasari, S.Kom., M.Sc., and the Vice Dean is Dimas Anton Asfani, S.T., M.T., Ph.D.",
    "In 2025, FT-EIC enrolled 1,360 new students, representing about 19.3% of ITS’s total intake.",
    "Noteworthy new students achieved top national scores, especially in Informatics Engineering.",

    # Universiti Malaya (UM)
    "Universiti Malaya (UM) is Malaysia's oldest university, located in Kuala Lumpur, with origins dating back to 1905.",
    "UM was formally established on October 8, 1949, from the merger of King Edward VII College of Medicine and Raffles College.",
    "UM's campus spans 922 acres (373 hectares), with over 27,000 students and more than 2,000 academic staff.",
    "UM offers a broad range of academic programs and is highly ranked in QS and THE Asia university rankings.",
    "UM has 14 faculties, two academies, three institutes, numerous research centers, and Malaysia's largest teaching hospital.",
    "The current Vice-Chancellor of UM is Professor Dato' Seri Ir. Dr. Noor Azuan Abu Osman."
]

In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
EMBEDDING_NAME = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBEDDING_NAME)

# Encode knowledge base
embeddings = embed_model.encode(knowledge_base, convert_to_numpy=True)

# Extract embedding dimensions
DIM = embed_model.get_sentence_embedding_dimension()

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 5: Create Qdrant Collection
- A **collection** in Qdrant is like a container (or folder) for storing vectors.  
- We must check if the collection already exists:
  - If it does not, we create a new one.  
- When creating, we configure how vectors will be stored:  
  - **Size** → number of values in each embedding (must match our embedding model dimensions).  
  - **Distance** → method for comparing similarity between vectors (we use **Cosine similarity**).  


In [ ]:
COLLECTION = "icts_its_um_minilm"
def ensure_collection():
    if not client.collection_exists(COLLECTION):
        client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
        )
ensure_collection()

# Step 6: Store Knowledge into Qdrant
- We already created a collection in Qdrant; now we need to **fill it with data**.  
- For each fact in our `knowledge_base`:
  - Convert the text into an **embedding vector**  
  - Wrap it into a `PointStruct`:
    - **id** → unique number for this point  
    - **vector** → the embedding (list of numbers)  
    - **payload** → original text (kept for easy retrieval later)  
- Collect all points into a list called `points`.  
- Use `client.upsert(...)` to **insert or update** these points into the Qdrant collection.  
  - `upsert` = update if the ID already exists, otherwise insert new.  


In [ ]:
points = [
    PointStruct(id=i, vector=embeddings[i].tolist(), payload={"text": knowledge_base[i]})
    for i in range(len(knowledge_base))
]
client.upsert(collection_name=COLLECTION, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

# Step 7: Retrieval

Now that our knowledge is stored in Qdrant, we need a way to **search and retrieve** the most relevant pieces when the user asks a question.

1. **`retrieve_context_qdrant(query, top_k)`**
   - Take the user’s question and turn it into an **embedding vector**.  
   - Ask Qdrant to **find the top_k most similar vectors** in the collection.  
   - Return a list of results, each containing:
     - **rank** → order of relevance  
     - **score** → similarity value (higher = more similar)  
     - **id** → point ID from Qdrant  
     - **text** → the original stored fact  

2. **`show_context_table(items)`**
   - Nicely display the retrieved results in a **table format** using Pandas.  
   - Includes rank, score, ID, and text.  
   - Rounds scores for easier reading.  

3. **`show_markdown_answer(query, answer_md)`**
   - Show the final answer in **Markdown formatting** (clean, easy-to-read).  
   - Also display the **original question** for context.  

Together, these functions let us:
- Retrieve the most relevant knowledge for a query  
- Display the supporting context clearly  
- Present the generated answer in a neat format  


In [ ]:
def retrieve_context_qdrant(query: str, top_k: int = 3) -> List[Dict[str, Any]]:
    qvec = embed_model.encode([query], convert_to_numpy=True)[0].tolist()
    res = client.query_points(
        collection_name=COLLECTION,
        query=qvec,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
        # optionally set a minimum similarity
        # score_threshold=0.0,
    )
    points = getattr(res, "points", res)
    items = []
    for rank, p in enumerate(points, start=1):
        items.append({
            "rank": rank,
            "score": float(p.score) if hasattr(p, "score") else None,
            "id": getattr(p, "id", None),
            "text": p.payload.get("text") if p.payload else None,
        })
    return items

In [ ]:
def show_context_table(items: List[Dict[str, Any]], title: str = "Retrieved Context"):
    if not items:
        display(Markdown(f"### {title}\n_No results returned._"))
        return
    df = pd.DataFrame(items, columns=["rank", "score", "id", "text"])
    # Tidy up score precision for readability
    if "score" in df.columns:
        df["score"] = df["score"].map(lambda x: round(x, 4) if isinstance(x, (int, float)) else x)
    display(Markdown(f"### {title} (k={len(items)})"))
    display(df)

def show_markdown_answer(query: str, answer_md: str):
    display(Markdown(f"# Answer\n\n{answer_md}"))
    display(Markdown(f"---\n_*Question:* **{query}**"))

# Step 8: Generation (Producing the Final Answer)

After retrieving the most relevant facts, we now pass them into a **Large Language Model (LLM)** to create a clear, natural answer.

1. **`generate_response(query, context_texts, model_name)`**
   - Takes the **user’s question** and the **retrieved context texts**.  
   - Builds a clean **prompt** for Gemini:
     - Puts the context into a **bullet list**  
     - Adds the user’s question below it  
     - Asks Gemini to give a concise Markdown answer  
   - Calls Gemini (`genai.GenerativeModel`) to generate the response text.  

2. **`rag_with_gemini(query, top_k)`**
   - The main “glue” function that connects everything together:
     - **Retrieve** top_k relevant items from Qdrant  
     - **Send** only their texts + the question to Gemini  
     - **Generate** an answer using `generate_response`  
     - **Display** the results in neat tables and Markdown  
   - Returns both:
     - **context** → the retrieved items  
     - **answer_markdown** → the generated answer  

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash"

In [ ]:
def generate_response(query, context_texts, model_name=GEMINI_MODEL):
    model = genai.GenerativeModel(model_name)
    # turn context into bullet list (cleaner prompt)
    bullets = "\n".join(f"- {c}" for c in context_texts if c)
    prompt = (
        f"Use only the context when possible; if unsure, say so.\n\n"
        f"## Context\n{bullets}\n\n"
        f"## Question\n{query}\n\n"
        f"## Answer (use clear, concise Markdown text):"
    )
    resp = model.generate_content(prompt)
    return resp.text

In [ ]:
def rag_with_gemini(query, top_k: int = 3, model_name=GEMINI_MODEL):
    # 1) retrieve structured items
    items = retrieve_context_qdrant(query, top_k=top_k)
    show_context_table(items, title="Retrieved Context")

    # 2) send only the texts to the model
    context_texts = [it["text"] for it in items if it.get("text")]
    answer_md = generate_response(query, context_texts, model_name=model_name)

    # 3) display pretty Markdown answer
    show_markdown_answer(query, answer_md)

    # also return pieces if you want to reuse programmatically
    return {"context": items, "answer_markdown": answer_md}


# Test RAG Pipeline

In [ ]:
QUESTION = "What can you tell me about ITS?"
results = rag_with_gemini(QUESTION, top_k=5)
items = results["context"]
answer_md = results["answer_markdown"]

### Retrieved Context (k=5)

,rank,score,id,text
0,1,0.3653,3,"ITS emphasizes science, engineering, technolog..."
1,2,0.1721,10,"The department has several research labs, incl..."
2,3,0.1638,13,The vision of FT-EIC is to be a leading facult...
3,4,0.1623,5,The Informatics Department (Departemen Teknik ...
4,5,0.1592,21,"UM has 14 faculties, two academies, three inst..."


NameError: name 'genai' is not defined

In [ ]:
print("Raw items:", items[:2])   # peek at first 2 retrieved
print("Answer text:", answer_md[:100], "...")  # peek at answer

Raw items: [{'rank': 1, 'score': 0.3652867, 'id': 3, 'text': 'ITS emphasizes science, engineering, technology education, and sustainable innovations in green technology and clean energy.'}, {'rank': 2, 'score': 0.17206019, 'id': 10, 'text': 'The department has several research labs, including the Software Engineering Lab, led by Prof. Umi Laili Yuhana.'}]
Answer text: ITS emphasizes science, engineering, technology education, and sustainable innovations in green tech ...
